# Walkthrough: from concept to steered model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvkevi/steerkit/blob/main/examples/walkthrough.ipynb)

This notebook is the unrolled version of the steerkit pipeline — every phase explained, every intermediate inspectable. Concept: **formality** (formal business tone vs. casual conversational). Model: **Qwen2.5-1.5B-Instruct** on MPS / CUDA / CPU (`steerkit.load` picks the device).

End-to-end runs in tens of seconds with the activation cache cold; near-instant when warm. Substitute any concept of your own once the loop is clear — `examples/data/` ships with verbosity, formality, refusal, and a multi-class emotion group; or call the teacher to generate fresh contrast pairs.

## Phase 0 — environment

On Colab the cell below clones the repo to `/content/steerkit` and pip-installs from there so the bundled `examples/data/` paths resolve. Locally it just resolves `REPO_ROOT` to the repo root.

The `TRANSFORMERLENS_ALLOW_MPS` flag silences a noisy MPS warning on Apple Silicon; harmless on CUDA / CPU.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_ROOT = Path("/content/steerkit")
    if not REPO_ROOT.exists():
        os.system(f"git clone -q https://github.com/arvkevi/steerkit {REPO_ROOT}")
    os.system(f"pip install -q {REPO_ROOT}")
else:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()

os.environ.setdefault("TRANSFORMERLENS_ALLOW_MPS", "1")  # harmless on CUDA / CPU

## Phase 1 — the labeled data

Steerkit trains on **contrast pairs**: the same prompt with two assistant responses, one that exhibits the concept and one that's neutral. The bundled `formality.jsonl` has 30 teacher-generated pairs. Each pair shares a benign user prompt; `positive_response` is formal/business-tone and `negative_response` is casual/conversational.

Why pairs and not single-class labels? Pairs hold everything-except-the-concept constant, so the resulting direction is a much cleaner signal-to-noise than "all the activations of category X vs. all the activations of category Y" where dataset bias contaminates the answer.

In [ ]:
from steerkit import load_pairs_jsonl

pairs = load_pairs_jsonl(REPO_ROOT / "examples" / "data" / "formality.jsonl")
print(f"loaded {len(pairs)} contrast pairs")
print()
print("first pair:")
p = pairs[0]
print(f"  prompt:            {p.prompt}")
print(f"  positive (formal): {p.positive_response[:120]}...")
print(f"  negative (casual): {p.negative_response[:120]}...")

## Phase 2 — load the target model

We use Qwen2.5-1.5B-Instruct: small enough to load and probe on MPS in a few seconds, large enough that tonal shifts (formal/casual, verbose/concise) are cleanly separable. The pipeline is architecture-agnostic — swap in any open-weight model TransformerLens supports.

In [ ]:
from steerkit import load

model = load("Qwen/Qwen2.5-1.5B-Instruct")
print(f"layers={model.n_layers}  d_model={model.d_model}  device={model.device}")

## Phase 3 — extract activations

For each (positive, negative) response we run a single forward pass and capture the residual stream at the last real token of every layer. Output: a `[n_pairs, 2, d_model]` tensor per layer (positive index 0, negative index 1).

`include_boundaries=False` skips the embedding + final-LN layers. Formality has a strong vocabulary signal at the embedding (formal words ARE different tokens than casual ones), so picking the embedding layer would be "trivially right" but uninformative — steering at layer 0 doesn't actually shape the model's reasoning, it only changes input representations.

`cache_dir` is a Zarr v3 activation cache keyed on `(model_id, hook_site, pairs_hash)`. The first run extracts; subsequent runs with the same inputs skip the model entirely.

In [ ]:
from steerkit import extract_activations

activations = extract_activations(
    pairs, model,
    hook_site="resid_post",
    include_boundaries=False,
    cache_dir=REPO_ROOT / "cache",
)
first_layer = next(iter(activations))
print(f"layers covered: {min(activations)}..{max(activations)}")
print(f"shape per layer: {tuple(activations[first_layer].shape)}  (n_pairs, {{pos,neg}}, d_model)")

## Phase 4 — fit probes per layer

Three candidate directions per layer, fit in one pass:

* **logistic regression** — sklearn's `LogisticRegression`. The default; best held-out classifier signal.
* **difference-of-means** — `mean(positives) − mean(negatives)`. Cheapest direction.
* **mass-mean / shrinkage LDA** — handles `d_model >> n_samples` regimes.

Each layer's `Probe` carries all three direction vectors plus held-out AUC and Cohen's d on the logistic decision function. The user picks which direction to use at steer-time via `probe.steer(..., method=...)`.

In [ ]:
from steerkit import Probe

probes = Probe.fit_all(activations, model, hook_site="resid_post", test_fraction=0.2)
# AUC saturates at 1.0 across many layers with a 30-pair dataset; Cohen's d on
# the logistic decision function is continuous and prefers mid-network layers
# where the concept is most cleanly *abstracted*, not reliant on token surface.
best = Probe.best_layer(probes, by="cohens_d_logistic")
print(f"best layer = {best.layer}  (depth {best.normalized_depth:.2f})")
print(f"AUC (test): {best.metrics['auc_test_logistic']:.3f}  Cohen's d: {best.metrics['cohens_d_logistic']:.2f}")
print(f"directions: {list(best.directions)}")

## Phase 5 — visualize the layer-selection curve

The layer-selection plot shows where the formality direction is most cleanly readable. With a small dataset like ours AUC saturates near 1.0 across most layers, so we plot Cohen's d instead — which is continuous and surfaces real layer-wise structure.

In [ ]:
from steerkit import plot_layer_selection

fig = plot_layer_selection(
    probes,
    by_classifier="cohens_d_logistic",
    x_axis="normalized_depth",
    title="formality probe across layers",
)
fig

## Phase 6 — calibrate α

`calibrate_alpha` sweeps a few candidate strengths and picks the largest α whose mean response perplexity stays within 1.5× of the unsteered baseline. This is a **coherence guardrail**, not a "did it work" check — it tells you the steered output isn't gibberish, nothing more.

Behavioral commitment to the concept usually requires 1.5-2× the calibrated α (see `docs/alpha_sweep.png`).

In [ ]:
from steerkit import calibrate_alpha

chosen, ratios = calibrate_alpha(best, model)
print(f"auto_alpha = {chosen:.3f}")
for a, r in sorted(ratios.items()):
    marker = "  ✓" if a == chosen else ""
    print(f"  α = {a:>6.2f}  ppl ratio = {r:.3f}{marker}")

## Phase 7 — save and reload

The `.probe.safetensors` artifact bundles tensors + JSON metadata: model id, hook name, layer, normalized depth, dataset hash, all three directions, biases, metrics, and the calibrated α. It travels in a single file.

In [ ]:
best.save(REPO_ROOT / "runs" / "formality.probe.safetensors")
reloaded = Probe.load(REPO_ROOT / "runs" / "formality.probe.safetensors")
print(f"reloaded probe trained on {reloaded.model_id}")
print(f"  layer: {reloaded.layer} (depth {reloaded.normalized_depth:.3f})")
print(f"  auto_alpha: {reloaded.auto_alpha:.3f}")

## Phase 8 — steer

`probe.steer(model, prompt)` returns a generated completion under the steering hook. Default `op="addition"` adds `α·v` to the residual stream at every generated token; `α=None` uses the calibrated `auto_alpha`.

We'll run an α sweep on the same prompt to see the strength dial in action.

In [ ]:
PROMPT = "What's a fun activity for a quiet Sunday afternoon?"
MAX_TOKENS = 40

for mult, label in [(0.0, "unsteered"), (1.0, "auto_α"), (2.0, "2× auto_α"), (4.0, "4× auto_α")]:
    alpha = mult * (best.auto_alpha or 8.0)
    out = best.steer(model, PROMPT, alpha=alpha, max_new_tokens=MAX_TOKENS)
    print(f"[α = {alpha:>6.2f}  ·  {label}]")
    print(f"  {out.strip()}")
    print()

### Per-token scoring

`probe.score_tokens(...)` projects every token's residual-stream activation at the probe's layer onto the probe direction. This is the interpretability complement to steering: it shows *where in a generated sequence* the formality direction is most active. Useful for sanity-checking that the hook is doing what you think it is.

In [ ]:
unsteered = best.steer(model, PROMPT, alpha=0.0, max_new_tokens=24)
steered = best.steer(model, PROMPT, alpha=2.0 * (best.auto_alpha or 8.0), max_new_tokens=24)

ts_unsteered = best.score_tokens(model, PROMPT, unsteered)
ts_steered = best.score_tokens(model, PROMPT, steered)

fig = ts_steered.plot(title=f"steered output (α = 2×auto_α)")
fig

### The other three operations

Addition is the default and the most common. Three alternatives — see `docs/ops_effects.png` for a side-by-side visual:

* `probe.ablate(model, prompt)` — projection: remove the concept's component entirely.
* `probe.clamp(model, prompt, target=8.0)` — force the concept's projection to a fixed value.
* `probe.amplify(model, prompt, gamma=4.0)` — scale the existing component by γ.

In [ ]:
for op_name, kwargs in [
    ("baseline (no hook)", {"alpha": 0.0}),
    ("addition (α = 2×auto_α)", {"alpha": 2.0 * (best.auto_alpha or 8.0)}),
    ("projection (ablate)", {"op": "projection"}),
    ("clamp at target=8", {"op": "clamp", "target": 8.0}),
    ("amplify γ=4", {"op": "multiplicative", "gamma": 4.0}),
]:
    out = best.steer(model, PROMPT, max_new_tokens=32, **kwargs)
    print(f"[{op_name}]")
    print(f"  {out.strip()}")
    print()

## Phase 9 — evaluate

Beyond the held-out classifier metrics computed during fitting, the `eval` module offers three more checks: a logit-lens vocabulary match, a perplexity ratio under steering, and an external-classifier shift if you supply one.

In [ ]:
from steerkit import evaluate_probe

report = evaluate_probe(
    best, model,
    target_vocab={"however", "furthermore", "consequently", "regards", "therefore"},
    perplexity_prompts=[PROMPT, "Recommend a book."],
)
print(report.summary())

## What's next

* **Pick your own concept**: edit a JSONL of contrast pairs or use the teacher pipeline (`generate_pairs_for_concept(...)`) — see `docs/concepts.md` for ideas.
* **Multi-concept**: `ConceptGroup` + `sweep(...)` produces a `GroupFit` with per-concept probes plus an optional joint multinomial classifier.
* **Compose**: `compose([probe_a, probe_b])` merges two single-concept directions for simultaneous steering.
* **Lint datasets**: `steerkit lint-pairs --pairs your.jsonl` runs eight cheap dataset-quality checks before you spend compute on a bad fit.
* **Refusal case study**: see `examples/case_studies/refusal_walkthrough.ipynb` for the unrolled refusal-probe variant of this workflow.